# Superstore Sales EDA: 01. Data Ingestion & Cleaning
---
**Objective:** Ingest raw transactional data, perform systematic schema validation, identify and address data quality anomalies (missing values, formatting, invalid types), and produce an immutable cleaned baseline for downstream analysis.

### Workflow:
1. Environment setup and library imports
2. Raw data loading and dimension audit
3. Missing value & duplicate detection
4. Postal code & geographic standardization
5. Temporal type casting & chronological validity checks
6. Numeric boundary & sanity constraints
7. Export cleaned dataset to `data/cleaned/superstore_cleaned.csv`


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Dynamic path resolution (supports both running from repo root or notebooks/)
def get_path(rel_path):
    for candidate in [Path(rel_path), Path("..") / rel_path, Path("/home/adarsh/Documents/Superstore Sales EDA") / rel_path]:
        if candidate.exists():
            return candidate
    return Path(rel_path)

raw_path = get_path("data/raw/Sample - Superstore.csv")
if not raw_path.exists():
    raw_path = get_path("data/raw/Superstore.csv")

print(f"Loading raw data from: {raw_path}")


Loading raw data from: data/raw/Sample - Superstore.csv


## 1. Load Raw Dataset & Inspect Structure
We load the raw CSV with `windows-1252` encoding to properly handle special characters in product names.


In [2]:
try:
    df_raw = pd.read_csv(raw_path, encoding="windows-1252")
except UnicodeDecodeError:
    df_raw = pd.read_csv(raw_path, encoding="utf-8")

print(f"Total Transactions: {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")
df_raw.head(3)


Total Transactions: 9,994
Total Columns: 21


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


## 2. Schema & Data Types Inspection
Examine columns, non-null counts, and data types across all 21 raw features.


In [3]:
df_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

## 3. Duplicate Records Audit
We check for exact duplicate rows as well as duplicate primary key indicators (`Row ID`).


In [4]:
exact_dupes = df_raw.duplicated().sum()
row_id_dupes = df_raw["Row ID"].duplicated().sum()

print(f"Exact row duplicates: {exact_dupes}")
print(f"Duplicate Row IDs: {row_id_dupes}")
assert exact_dupes == 0, "Warning: Duplicate rows detected!"
assert row_id_dupes == 0, "Warning: Duplicate Row IDs detected!"


Exact row duplicates: 0
Duplicate Row IDs: 0


## 4. Missing Value Audit & Postal Code Treatment
In retail datasets, missing ZIP codes or geographic nulls can distort regional reporting.
We check for nulls and format postal codes as standardized 5-digit strings.


In [5]:
null_summary = df_raw.isnull().sum()
print("Missing values per column:")
print(null_summary[null_summary > 0] if (null_summary > 0).any() else "No NaN values detected.")

df_clean = df_raw.copy()

# Standardize Postal Code
# Impute Burlington, VT if missing or 0
burlington_mask = (df_clean["City"] == "Burlington") & (df_clean["State"] == "Vermont")
df_clean.loc[burlington_mask & df_clean["Postal Code"].isnull(), "Postal Code"] = 5401

df_clean["Postal Code"] = (
    df_clean["Postal Code"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(5)
)
df_clean.loc[df_clean["Postal Code"] == "00000", "Postal Code"] = "Unknown"

print("Postal code formatted sample:")
print(df_clean[['City', 'State', 'Postal Code']].head(4))


Missing values per column:
No NaN values detected.
Postal code formatted sample:
              City       State Postal Code
0        Henderson    Kentucky       42420
1        Henderson    Kentucky       42420
2      Los Angeles  California       90036
3  Fort Lauderdale     Florida       33311


## 5. Temporal Conversions & Logical Validation
Convert `Order Date` and `Ship Date` to `datetime64[ns]`.
Verify the business rule: **`Ship Date >= Order Date`** (no orders shipped prior to purchase).


In [6]:
df_clean["Order Date"] = pd.to_datetime(df_clean["Order Date"], format="mixed")
df_clean["Ship Date"] = pd.to_datetime(df_clean["Ship Date"], format="mixed")

# Validation check
invalid_lead_times = (df_clean["Ship Date"] < df_clean["Order Date"]).sum()
print(f"Orders with Ship Date < Order Date: {invalid_lead_times}")
assert invalid_lead_times == 0, "Temporal anomaly: Dispatch date precedes order date!"

print(f"Earliest Order Date: {df_clean['Order Date'].min().strftime('%Y-%m-%d')}")
print(f"Latest Order Date:   {df_clean['Order Date'].max().strftime('%Y-%m-%d')}")


Orders with Ship Date < Order Date: 0
Earliest Order Date: 2014-01-03
Latest Order Date:   2017-12-30


## 6. Financial & Numeric Bounds Validation
Check for impossible business values:
- `Sales > 0`
- `Quantity >= 1`
- `0.0 <= Discount <= 1.0`


In [7]:
sales_check = (df_clean["Sales"] <= 0).sum()
quantity_check = (df_clean["Quantity"] <= 0).sum()
discount_check = ((df_clean["Discount"] < 0) | (df_clean["Discount"] > 1.0)).sum()

print(f"Invalid Sales (<=0): {sales_check}")
print(f"Invalid Quantity (<=0): {quantity_check}")
print(f"Invalid Discount (<0 or >1): {discount_check}")

assert sales_check == 0 and quantity_check == 0 and discount_check == 0, "Numeric integrity violation!"

# Summary statistics of numerical columns
df_clean[["Sales", "Quantity", "Discount", "Profit"]].describe().T


Invalid Sales (<=0): 0
Invalid Quantity (<=0): 0
Invalid Discount (<0 or >1): 0


,count,mean,std,min,25%,50%,75%,max
Sales,9994.0,229.858001,623.245101,0.444,17.28000,54.4900,209.940,22638.480
Quantity,9994.0,3.789574,2.225110,1.000,2.00000,3.0000,5.000,14.000
Discount,9994.0,0.156203,0.206452,0.000,0.00000,0.2000,0.200,0.800
Profit,9994.0,28.656896,234.260108,-6599.978,1.72875,8.6665,29.364,8399.976


## 7. Categorical Standardization & Export
Strip extraneous whitespace from string/categorical dimensions and persist cleaned dataset.


In [8]:
str_cols = ["Order ID", "Customer ID", "Customer Name", "Country", 
            "City", "State", "Region", "Product ID", "Category", 
            "Sub-Category", "Product Name", "Ship Mode", "Segment"]

for col in str_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()

out_dir = Path("data/cleaned") if Path("data").exists() else Path("../data/cleaned")
out_dir.mkdir(parents=True, exist_ok=True)
out_clean = out_dir / "superstore_cleaned.csv"

df_clean.to_csv(out_clean, index=False)
print(f"Successfully cleaned and exported {len(df_clean):,} records to: {out_clean}")


Successfully cleaned and exported 9,994 records to: data/cleaned/superstore_cleaned.csv
